In [ ]:
! pip install sewar
# wignerdpy à installer manuellement : GithHub ljbkusters/python-wigner-distribution

In [ ]:
import rasterio
from rasterio.enums import Resampling

from skimage.transform import resize

#from sewar.full_ref import rmse
from sewar.full_ref import psnr
from sewar.full_ref import ssim
#from sewar.full_ref import scc
import pandas as pd

import os
import glob as glob
import numpy as np
import rasterio
from rasterio import plot
from rasterio.plot import show
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.cm import get_cmap
from matplotlib.axes import Axes
from IPython.display import HTML
from skimage import exposure
import rasterio
from rasterio.plot import show
import geopandas as gpd
import fiona
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import sklearn
from sklearn.metrics import classification_report,accuracy_score

from rasterio.warp import reproject, Resampling, calculate_default_transform



In [ ]:
BANDE_SR = "blue" # valeurs possibles "blue", "green", "red" et "nir"

RESOLUTION_S2 = 10
RESOLUTION_Planet = 2.5
UPSCALE_FACTOR = RESOLUTION_S2/RESOLUTION_Planet
print("UPSCALE FACTOR", UPSCALE_FACTOR)

REPERTOIRE_DATASET = "D:\\DATA\\FOLDER"
REPERTOIRE_SATELLITE_S2 = "S-2_DECOUPE_FINAL.OK"
REPERTOIRE_PLANET = "Planet_DECOUPE_FINAL.OK"
REPERTOIRE_SR = "SR" + "\\" + "SR_" + BANDE_SR
REPERTOIRE_COMPLET_SATELLITE_S2 = REPERTOIRE_DATASET + "\\" + REPERTOIRE_SATELLITE_S2
REPERTOIRE_COMPLET_SATELLITE_PLANET = REPERTOIRE_DATASET + "\\" + REPERTOIRE_PLANET
REPERTOIRE_COMPLET_SR = REPERTOIRE_DATASET + "\\" + REPERTOIRE_SR
EXTENSION_SENTINEL2 = ".jp2"
EXTENSION_PLANET = ".tif"
EXTENSION_SR = ".tif"

print("Bande Super Resolution ->", BANDE_SR)
print("REPERTOIRE COMPLET S2 ->", REPERTOIRE_COMPLET_SATELLITE_S2)
print("REPERTOIRE COMPLET Planet ->", REPERTOIRE_COMPLET_SATELLITE_PLANET)
print("REPERTOIRE COMPLET Super Resolution ->", REPERTOIRE_COMPLET_SR)
print("Extension S2 ->", EXTENSION_SENTINEL2)
print("Extension Planet ->", EXTENSION_PLANET)
print("Extension Super Resolution ->", EXTENSION_SR)

In [ ]:
def normalise(array):
    array_min, array_max = array.min(), array.max()
    return((array - array_min)/(array_max - array_min))

# utilisation de df.append, bien que déconseillée, car il y a peu de lignes
df_linear_psnr_ssim = pd.DataFrame(columns = ['PSNR', 'SSIM', 'CS'])
df_bicubic_psnr_ssim = pd.DataFrame(columns = ['PSNR', 'SSIM', 'CS'])

# création du dossier SR
if not os.path.exists(REPERTOIRE_COMPLET_SR):
    os.mkdir(REPERTOIRE_COMPLET_SR)

listes_id = os.listdir(REPERTOIRE_COMPLET_SATELLITE_S2)
num_obs = 0
for id in listes_id:
    print("Observation en cours: {}".format(id))

    ligne_linear = []
    ligne_bicubic = []
    df_tmp_linear = pd.DataFrame(columns = ['PSNR', 'SSIM', 'CS'])
    df_tmp_bicubic = pd.DataFrame(columns = ['PSNR', 'SSIM', 'CS'])
    
    # Récupération image Sentinel-2 de base
    print("Récupération images de base")
    # liste des images du repertoire S-2
    repertoire_image_S2 = REPERTOIRE_COMPLET_SATELLITE_S2 + "\\" + str(id)
    liste_images_sat_S2 = os.listdir(repertoire_image_S2)
    for image_S2 in liste_images_sat_S2:
        split_S2 = os.path.splitext(image_S2)
        nom_fichier_S2 = split_S2[0]
        extension_S2 = split_S2[1]
        if extension_S2 == EXTENSION_SENTINEL2:
            nom_fichier_decoupe_S2 = nom_fichier_S2.split("_")
            bande_S2 = nom_fichier_decoupe_S2[-1]
            if bande_S2 == "B02" and BANDE_SR == "blue":
                # Sentinel-2
                print("Sentinel-2")
                chemin_S2_original = repertoire_image_S2 + "\\" + image_S2
            if bande_S2 == "B03" and BANDE_SR == "green":
                # Sentinel-2
                print("Sentinel-2")
                chemin_S2_original = repertoire_image_S2 + "\\" + image_S2
            if bande_S2 == "B04" and BANDE_SR == "red":
                # Sentinel-2
                print("Sentinel-2")
                chemin_S2_original = repertoire_image_S2 + "\\" + image_S2
            if bande_S2 == "B08" and BANDE_SR == "nir":
                # Sentinel-2
                print("Sentinel-2")
                chemin_S2_original = repertoire_image_S2 + "\\" + image_S2

    # Lecture bande correspondante Planet et Transformation des images en array
    print("Lecture bande", BANDE_SR, "Planet")
    repertoire_image_planet = REPERTOIRE_COMPLET_SATELLITE_PLANET + '\\' + str(id)
    liste_images_sat_planet = os.listdir(repertoire_image_planet)
    # lecture des bandes suivant les fichiers correspondants
    for image_planet in liste_images_sat_planet:
        split_planet = os.path.splitext(image_planet)
        nom_fichier_planet = split_planet[0]
        extension_planet = split_planet[1]
        
        if extension_planet == EXTENSION_PLANET:
            fichier_img_planet = nom_fichier_planet.split("_")
            caracteristique_img_planet = fichier_img_planet[-2]
            if caracteristique_img_planet == 'SR':
                #lecture du fichier image
                chemin_image_planet = repertoire_image_planet + "\\" + image_planet
                img_planet = rasterio.open(chemin_image_planet)
    print("Transformation des images Planet en array suivant la bande", BANDE_SR)
    if BANDE_SR == "blue":
        planet_array = img_planet.read(1)
    elif BANDE_SR == "green":
        planet_array = img_planet.read(2)
    elif BANDE_SR == "red":
        planet_array = img_planet.read(3)
    elif BANDE_SR == "nir":
        planet_array = img_planet.read(4)
    else:
        print("Mauvais choix de bande")

    
    rep_target_upsampled = REPERTOIRE_COMPLET_SR + "\\" + str(id)

    # création des dossier id dans les repertoires resampled et coregistered
    if not os.path.exists(rep_target_upsampled):
        os.mkdir(rep_target_upsampled)

    im_target_sr_linear = rep_target_upsampled + "\\" + str(id) + "_sentinel2_sr_linear" + EXTENSION_SR
    im_target_sr_bicubic = rep_target_upsampled + "\\" + str(id) + "_sentinel2_sr_bicubic" + EXTENSION_SR
    
    # SR with linear + bicubic
    with rasterio.open(chemin_S2_original) as image_to_sr:
        # resample data to target shape Linear Interpolation
        img_S2_sr_linear = image_to_sr.read(
            out_shape=(
                image_to_sr.count,
                int(image_to_sr.height * UPSCALE_FACTOR),
                int(image_to_sr.width * UPSCALE_FACTOR)
            ),
            resampling=Resampling.bilinear
        )
        # resample data to target shape Bicubic Interpolation
        img_S2_sr_bicubic = image_to_sr.read(
            out_shape=(
                image_to_sr.count,
                int(image_to_sr.height * UPSCALE_FACTOR),
                int(image_to_sr.width * UPSCALE_FACTOR)
            ),
            resampling=Resampling.cubic
        )
    
        # scale image transform
        transform_linear = image_to_sr.transform * image_to_sr.transform.scale(
            (image_to_sr.width / img_S2_sr_linear.shape[-1]),
            (image_to_sr.height / img_S2_sr_linear.shape[-2])
        )
        transform_bicubic = image_to_sr.transform * image_to_sr.transform.scale(
            (image_to_sr.width / img_S2_sr_bicubic.shape[-1]),
            (image_to_sr.height / img_S2_sr_bicubic.shape[-2])
        )

    # Metada update
    print("Mise à jour des métadonnées par rapport à celles de Planet")
    meta_linear = image_to_sr.meta
    meta_linear.update({"height": img_planet.meta["height"]})
    meta_linear.update({"width": img_planet.meta["width"]})
    meta_linear.update({"transform": transform_linear})
    meta_bicubic = image_to_sr.meta
    meta_bicubic.update({"height": img_planet.meta["height"]})
    meta_bicubic.update({"width": img_planet.meta["width"]})
    meta_bicubic.update({"transform": transform_bicubic})

    print("Fermeture du fichier image Planet")
    img_planet.close()

    # Write super resolved image files
    print("Enregistrement des fichiers Super Resolution")
    with rasterio.open(im_target_sr_linear, 'w', **meta_linear) as dest1:
        dest1.write(img_S2_sr_linear)
    with rasterio.open(im_target_sr_bicubic, 'w', **meta_bicubic) as dest2:
        dest2.write(img_S2_sr_bicubic)

    # Normalisation pour comparer valeurs d'échelles différentes
    print("Normalisation")
    planet_array_norm = normalise(planet_array)
    img_S2_sr_linear_norm = normalise(img_S2_sr_linear)
    img_S2_sr_bicubic_norm = normalise(img_S2_sr_bicubic)

    # Metrics
    print("Calcul des métriques")
#    img_source = planet_array_norm
#    img_target_linear = resize(img_S2_sr_linear[0], img_source.shape)
#    img_target_bicubic = resize(img_S2_sr_bicubic[0], img_source.shape)  
    img_source = planet_array_norm
    img_target_linear = resize(img_S2_sr_linear_norm[0], img_source.shape)
    img_target_bicubic = resize(img_S2_sr_bicubic_norm[0], img_source.shape)
#    max_planet = img_source.max().item()
#    max_linear = img_target_linear.max().item()
#    max_bicubic = img_target_bicubic.max().item()
#    max_val_datarange = max(max_planet, max_linear, max_bicubic)
    max_val_datarange = 1 # The max possible value of a normalized pixel is 1
    psnr_linear = psnr(img_source, img_target_linear, MAX = max_val_datarange)
    psnr_bicubic = psnr(img_source, img_target_bicubic, MAX = max_val_datarange)
    (ssim_linear, cs_linear) = ssim(img_source, img_target_linear, MAX = max_val_datarange)
    (ssim_bicubic, cs_bicubic) = ssim(img_source, img_target_bicubic, MAX = max_val_datarange)
    
    # Ajout
    print("Enegistrement des nouvelles valeurs de métriques")
    print("Linear")
    ligne_linear.append(psnr_linear)
    ligne_linear.append(ssim_linear)
    ligne_linear.append(cs_linear)
    df_tmp_linear.loc[0] = ligne_linear
    df_linear_psnr_ssim = pd.concat([df_linear_psnr_ssim, df_tmp_linear], ignore_index = True)
    print("Bicubic")
    ligne_bicubic.append(psnr_bicubic)
    ligne_bicubic.append(ssim_bicubic)
    ligne_bicubic.append(cs_bicubic)
    df_tmp_bicubic.loc[0] = ligne_bicubic
    df_bicubic_psnr_ssim = pd.concat([df_bicubic_psnr_ssim, df_tmp_bicubic], ignore_index = True)

    # Affichage
    print("Affichage des valeurs de métriques pour l'observation", id)
    print("ALGO: LINEAR")
    print("PSNR", psnr_linear, ". SSIM", ssim_linear, ". CS", cs_linear)
    print("ALGO: BICUBIC")
    print("PSNR", psnr_bicubic, ". SSIM", ssim_bicubic,". CS", cs_bicubic)

    num_obs += 1
    print("") # saut de ligne pour séparation messages par observation

print("Nombre d'observations traitées au total", num_obs)

In [ ]:
# TEST
img_source = planet_array_norm
img_target_linear = resize(img_S2_sr_linear_norm[0], img_source.shape)
img_target_bicubic = resize(img_S2_sr_bicubic_norm[0], img_source.shape)
max1 = img_source.max().item()
max2 = img_target_linear.max().item()
max3 = img_target_bicubic.max().item()
max_val_datarange = max(max1, max2, max3)

In [ ]:
# TEST
#MAX = np.iinfo(img_source.dtype).max
print(img_source.dtype)
print(img_target_linear.dtype)
print(img_target_bicubic.dtype)

In [ ]:
# TEST
#psnr_linear = ssim(img_source,img_target_linear)
#psnr_bicubic = ssim(img_source,img_target_bicubic)
#psnr_bicubic = psnr(img_source, img_target_bicubic, MAX = max_val_datarange)
#ssim_linear = ssim(img_source, img_target_linear)
ssim_bicubic, ssim_sc_bicubic = ssim(img_source, img_target_bicubic, MAX = max_val_datarange)

# TODO

#print(ssim_linear[1])
#print(psnr_bicubic)

print(ssim_bicubic, ssim_sc_bicubic)
print(img_source)
print(img_target_linear)
print(img_target_bicubic)

In [ ]:
# TEST
print("Normalisation")
planet_array_norm = normalise(planet_array)
img_S2_sr_linear_norm = normalise(img_S2_sr_linear)
img_S2_sr_bicubic_norm = normalise(img_S2_sr_bicubic)

# Metrics
print("Calcul des métriques")
#    img_source = planet_array_norm
#    img_target_linear = resize(img_S2_sr_linear[0], img_source.shape)
#    img_target_bicubic = resize(img_S2_sr_bicubic[0], img_source.shape)  
img_source = planet_array_norm
img_target_linear = resize(img_S2_sr_linear_norm[0], img_source.shape)
img_target_bicubic = resize(img_S2_sr_bicubic_norm[0], img_source.shape)
max_planet = img_source.max().item()
max_linear = img_target_linear.max().item()
max_bicubic = img_target_bicubic.max().item()
max_val_datarange = max(max_planet, max_linear, max_bicubic)
psnr_linear = psnr(img_source, img_target_linear, MAX = max_val_datarange)
psnr_bicubic = psnr(img_source, img_target_bicubic, MAX = max_val_datarange)
(ssim_linear, cs_linear) = ssim(img_source, img_target_linear, MAX = max_val_datarange)
(ssim_bicubic, cs_bicubic) = ssim(img_source, img_target_bicubic, MAX = max_val_datarange)


In [ ]:
print("Results algo LINEAR, band =", BANDE_SR)
print(df_linear_psnr_ssim.mean())
print("Results algo BICUBIC, band =", BANDE_SR)
print(df_bicubic_psnr_ssim.mean())